<center>
<h1 style="font-family:verdana">
 🏘 Sistema de diàleg basat en regles 🏘 <h1>








<p>   🎯 <b>Objectiu</b>: en aquesta pràctica aprendrem a crear un assistent senzill de compra d'habitatges. El sistema haurà d'identificar i mostrar les cases que coincideixen amb les preferències de l'usuari. A poc a poc anirem afegint-hi funcionalitats, perquè el sistema siga més complet. </p>

<p> ✨ <b>Contingut</b>: en primer lloc, començarem amb un exemple senzill en què el sistema llançarà unes preguntes i respostes predefinides i l'usuari haurà d'escollir. A poc a poc anirem afegint-hi funcionalitats, perquè el sistema siga més complet. </p>

✏ <b>Exercicis</b>: en cada secció anireu trobant exercicis que haureu d'anar resolent.

---

<h2> Índex </h2>


1. [Fitxer JSON](#section-one)
  * [Exercici 1](#ex-one)
2. [Sistema de diàleg senzill](#section-two)
  * [Exercici 2](#ex-two)
  * [Exercici 3](#ex-three)
  * [Exercici 4](#ex-four)
3. [Millorem el sistema de diàleg](#section-three)
  * [Exercici 5](#ex-five)
  * [Exercici 6](#ex-six)
4. [Lliurable](#section-four)
---


In [19]:
from nltk.tokenize.treebank import TreebankWordTokenizer
import nltk
nltk.download('omw-1.4')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

import json
import sys

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


<h1><a name="section-one"> 1. Fitxer JSON </a></h1>

En primer lloc, carregarem el fitxer JSON per analitzar-ne l'estructura.

In [20]:
with open('base_dades_2.json') as f:
  data = json.load(f)

print('Start message: ', data['start_message'])
print('End message: ', data['end_message'])
print('Questions: ', data['questions'])
print('House: ', data['houses'])

Start message:  Welcome to the House Buying Assistant!
End message:  Thank you for using the House Buying Assistant. Goodbye!
Questions:  [{'question': 'How many bedrooms do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (1 - 5 bedrooms): ', 'answer_key': 'bedrooms'}, {'question': 'How many bathrooms do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (1 - 3 bathrooms): ', 'answer_key': 'bathrooms'}, {'question': 'What is your budget for the house?', 'type': 'numerical', 'prompt': 'Enter your choice:', 'answer_key': 'price'}, {'question': 'How many square meters do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (30 - 200 meters)', 'answer_key': 'square_meters'}, {'question': 'Which city or neighborhood would you prefer?', 'type': 'multichoice', 'prompt': 'Enter your choice', 'answer_key': 'location'}]
House:  [{'id': 1, 'type': 'rent', 'bedrooms': '3', 'bathrooms': '2', 'price': '1200', 'square_meters': '100', 'floor': '3', 'elevator': 'Yes', 

Podeu veure com al fitxer JSON podem trobar el missatge de benvinguda i el missatge de comiat. Les diferents preguntes predefinides que el sistema demanarà a l'usuari. I les cases que l'agència té disponibles per a oferir a l'usuari.

Si no acabeu d'entendre el format, podeu copiar el contingut del fitxer JSON [aquí](https://jsonviewer.stack.hu/) per visualitzar millor l'estructura.

---



 <h1><a name="ex-one"><center> ✏ Exercici 1 ✏</a></h1>


En aquest primer exercici us demanem que afegiu una nova casa manualment al fitxer JSON amb l'id 26. La resta de camps els podeu emplenar com vulgueu.

Si ho heu fet correctament podreu veure al següent *print* les dades que heu introduït.

In [21]:
data['houses'].append({
    "id": 26,
    "type": "sale",
    "housing_type": "adosada",
    "bedrooms": "4",
    "bathrooms": "2",
    "price": "250k",
    "square_meters": "140",
    "floor": "0",
    "elevator": "No",
    "commercial_use": "No",
    "terrace": "Yes",
    "location": "Montcada i Reixac"
})

id = 26
house = data['houses'][id-1]


print(f"House ID: {house['id']}")
print(f"Bedrooms: {house['bedrooms']}")
print(f"Bathrooms: {house['bathrooms']}")
print(f"Price: {house['price']}")
print(f"Square Meters: {house['square_meters']}")
print(f"Location: {house['location']}")

House ID: 26
Bedrooms: 4
Bathrooms: 2
Price: 250k
Square Meters: 140
Location: Montcada i Reixac


---

<h1><a name="section-two"> 2. Sistema de diàleg senzill </a></h1>


En aquesta primera part crearem un sistema senzill que mostrarà les preguntes predefinides i, a continuació l'usuari haurà d'introduir la resposta. Tractarem de manera diferent les preguntes en què la resposta siga un número i les preguntes d'opció múltiple.

Intenteu entendre el funcionament, ja que treballarem sobre aquest codi.

In [22]:
def print_question(prompt, possible_options = []):
    print(prompt)
    if not len(possible_options) == 0:
      print("Options:", ", ".join(possible_options))

def initialize_available_options(house_data, available_options):
    for house in house_data['houses']:
        for key, value in house.items():
            available_options.setdefault(key, set()).add(value)

def preprocess_answer(answer):
    answer = nltk.word_tokenize(answer)
    return answer

def get_numerical_value(tok_answer):
    for token in tok_answer:
        if token.isnumeric() or token[:-1].isnumeric():
            return token
    return ''

#Exemple de frase tokenitzada
print(preprocess_answer('Vull un pis de 35k €'))

['Vull', 'un', 'pis', 'de', '35k', '€']


In [23]:
def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt'])
      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if not value == '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt'])
      if answer in options:
        return answer

def chatbot_v1():
  user_preferences, available_options = {}, {}
  initialize_available_options(data, available_options)
  
  for question in data['questions']:
    answer_key = question['answer_key']
    possible_options = list(available_options.get(answer_key))

    if question['type'] == 'numerical':
      answer = process_numerical_question(question)
    else:
      answer = process_multichoice_question(question, possible_options)

    user_preferences[answer_key] = answer



---


 <h1><a name="ex-two"><center> ✏ Exercici 2 ✏ </a></h1>

A continuació millorarem un poc el sistema. Us proposem que afegiu la frase de benvinguda que tenim guardada al fitxer JSON. És a dir, abans de mostrar la primera pregunta el sistema ens donarà la benvinguda.

In [15]:
print(data['start_message'])
chatbot_v1()
print(data['end_message'])

Welcome to the House Buying Assistant!
How many bedrooms do you need?


KeyboardInterrupt: Interrupted by user

---


 <h1><a name="ex-three"><center> ✏ Exercici 3 ✏ </a></h1>

En aquest exercici us demanem que permeteu a l'usuari abandonar el programa quan ho desitge. Per exemple, si l'usuari escriu `quit`, que el sistema s'acomiade amb l'oració de comiat que trobareu al JSON i finalitze el programa.


🙃 `sys.exit()` genera una excepció SystemExit.

In [20]:
def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt'])

      if is_exit_command(answer):
        return "quit"
      
      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if not value == '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt'])
      
      if is_exit_command(answer):
        return "quit"
      
      if answer in options:
        return answer

def is_exit_command(answer):
    return answer.strip().lower() in {"q", "quit"}

In [8]:
def chatbot_v2():
  print(data['start_message'])
  user_preferences, available_options = {}, {}
  initialize_available_options(data, available_options)
  
  for question in data['questions']:
    answer_key = question['answer_key']
    possible_options = list(available_options.get(answer_key))

    if question['type'] == 'numerical':
      answer = process_numerical_question(question)
    else:
      answer = process_multichoice_question(question, possible_options)

    if answer == 'quit':
      print(print(data['end_message']))
      sys.exit()

    user_preferences[answer_key] = answer

  print(data['end_message'])

  return user_preferences


In [9]:
chatbot_v2()

Welcome to the House Buying Assistant!
How many bedrooms do you need?
Thank you for using the House Buying Assistant. Goodbye!
None


SystemExit: 

c:\Users\ashve\IA\q5\TVD\P1-SistemasBasadosEnReglas-TVD\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Funciona també si escriviu `Quit`?

---

 <h1><a name="ex-four"><center> ✏ Exercici 4 ✏ </a></h1>

Fins ara tenim a `user_preferences` les opcions que l'usuari ha escollit.







In [10]:
user_preference = chatbot_v2()


Welcome to the House Buying Assistant!
How many bedrooms do you need?
How many bathrooms do you need?
What is your budget for the house?
How many square meters do you need?
Which city or neighborhood would you prefer?
Options: Montcada i Reixac, L'Hospitalet de Llobregat, Barcelona, Esplugues de Llobregat, Santa Coloma de Gramenet
Thank you for using the House Buying Assistant. Goodbye!


Per tant, ara haurem de revisar si l'agència disposa d'alguna casa disponible amb aquestes característiques. Per això us proposem completar la funció `find_suitable_houses`. Aquesta funció tornarà una llista amb tants diccionaris com cases disponibles. Per exemple si segons les preferències de l'usuari només està disponible la casa amb id 3, `suitable_house` serà:


```
[{'id': 3, 'bedrooms': '2', 'bathrooms': '1', 'price': '160k', 'square_meters': '80', 'location': 'Santa Coloma de Gramenet'}]
```

In [11]:
def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt'])

      if is_exit_command(answer):
        return "quit"
      
      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if not value == '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt'])
      
      if is_exit_command(answer):
        return "quit"
      
      if answer in options:
        return answer

def is_exit_command(answer):
    return answer.strip().lower() in {"q", "quit"}

In [18]:
def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt'])

      if is_exit_command(answer):
        return "quit"
      
      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if not value == '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt'])
      
      if is_exit_command(answer):
        return "quit"
      
      if answer in options:
        return answer

def is_exit_command(answer):
    return answer.strip().lower() in {"q", "quit"}

def ask_user_preferences():
    user_preferences, available_options = {}, {}
    initialize_available_options(data, available_options)
    
    for question in data['questions']:
      answer_key = question['answer_key']
      possible_options = list(available_options.get(answer_key))


      if question['type'] == 'numerical':
        answer = process_numerical_question(question)
      else:
        answer = process_multichoice_question(question, possible_options)

      if answer == 'quit':
        print(data['end_message'])
        sys.exit()

      user_preferences[answer_key] = answer

    return user_preferences

def find_suitable_houses(data, user_preferences):
    suitable_houses = []

    for house in data['houses']:
        matches_preferences = all(
            house.get(key) == value
            for key, value in user_preferences.items()
        )

        if matches_preferences:
            suitable_houses.append(house)

    return suitable_houses

---



Una vegada tingueu la funció `find_suitable_houses` programada, podreu executar el següent *script* que us mostrarà, si n'hi ha, la casa o cases disponibles amb les característiques que heu triat.

In [24]:
def print_suitable_houses(suitable_houses):
  if suitable_houses:
    print("\nBased on your preferences, the most suitable houses are:")
    for house in suitable_houses:
      print(f"House ID: {house['id']}")
      print(f"Type: {house['type']}")
      print("Bedrooms:", house['bedrooms'])
      print("Bathrooms:", house['bathrooms'])
      print("Price:", house['price'], "euros")
      print("Square Meters:", house['square_meters'], "m^2")
      print("Location:", house['location'])
      print()
  else:
    print("\nSorry, no suitable houses match your preferences. \n")

In [14]:

def chatbot_v3():
  print(data['start_message'])
  user_preferences = ask_user_preferences()
  suitable_houses = find_suitable_houses(data, user_preferences)
  print_suitable_houses(suitable_houses)
  print(data['end_message'])


In [15]:
chatbot_v3()

Welcome to the House Buying Assistant!
How many bedrooms do you need?
How many bathrooms do you need?
What is your budget for the house?
How many square meters do you need?
Which city or neighborhood would you prefer?
Options: Barcelona, Esplugues de Llobregat, L'Hospitalet de Llobregat, Montcada i Reixac, Santa Coloma de Gramenet
Thank you for using the House Buying Assistant. Goodbye!


SystemExit: 

<h1><a name="section-three"> 3. Millorem el sistema de diàleg </a></h1>


---


 <h1><a name="ex-five"><center> ✏ Exercici 5 ✏ </a></h1>

Tal com haureu comprovat sempre esteu obligats a triar una opció. En aquest últim exercici proposem que afegiu l'opció de no triar-ne cap. És a dir, si ens és igual el nombre d'habitacions, podrem posar per exemple `any` i el sistema ens farà la següent pregunta.

In [16]:
def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt']).strip()

      if is_exit_command(answer):
        return "quit"

      if is_any_command(answer):
        return "any"

      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if value != '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt']).strip()

      if is_exit_command(answer):
        return "quit"

      if is_any_command(answer):
        return "any"

      if answer in options:
        return answer

def is_exit_command(answer):
    return answer.strip().lower() in {"q", "quit"}

def is_any_command(answer):
    return answer.strip().lower() == "any"

def ask_user_preferences():
    user_preferences, available_options = {}, {}
    initialize_available_options(data, available_options)

    for question in data['questions']:
      answer_key = question['answer_key']
      possible_options = list(available_options.get(answer_key))


      if question['type'] == 'numerical':
        answer = process_numerical_question(question)
      else:
        answer = process_multichoice_question(question, possible_options)

      if answer == 'quit':
        print(data['end_message'])
        sys.exit()

      user_preferences[answer_key] = answer

    return user_preferences

def find_suitable_houses(data, user_preferences):
    suitable_houses = []

    for house in data['houses']:
        matches_preferences = all(
            value == 'any' or house.get(key) == value
            for key, value in user_preferences.items()
        )

        if matches_preferences:
            suitable_houses.append(house)

    return suitable_houses

In [17]:
def chatbot_v4():
  print(data['start_message'])
  user_preferences = ask_user_preferences()
  suitable_houses = find_suitable_houses(data, user_preferences)
  print_suitable_houses(suitable_houses)
  print(data['end_message'])


In [51]:
chatbot_v4()

Welcome to the House Buying Assistant!
How many bedrooms do you need?
How many bathrooms do you need?
What is your budget for the house?
How many square meters do you need?
Which city or neighborhood would you prefer?
Options: Barcelona, Esplugues de Llobregat, L'Hospitalet de Llobregat, Santa Coloma de Gramenet
Are you looking to buy or rent a house?
Options: rent, sale
What is your household's monthly income?
What is the minimum floor you would like to live on?
Do you want a house with a terrace?
Options: No, Yes
Do you want a house with an elevator?
Options: No, Yes
Will you use the house for commercial purposes?
Options: No, Yes
Which city or neighborhood would you prefer?
Options: Barcelona, Esplugues de Llobregat, L'Hospitalet de Llobregat, Santa Coloma de Gramenet

Sorry, no suitable houses match your preferences. 

Thank you for using the House Buying Assistant. Goodbye!


---



 <h1><a name="ex-six"><center> ✏ Exercici 6 ✏ </a></h1>

Amplieu el vostre xatbot perquè tinga en compte la següent informació sobre:

1. **Tipus d'habitatge:** Pregunteu a l'usuari si vol una casa per a compra (tipus *"sale"* a les dades) o per llogar (tipus *"rent"*). Al final de l'execució, el sistema haurá de mostrar habitatges amb el tipus seleccionat.

2. **Ingressos:** S'aconsella no dedicar més del 35% dels ingressos a l'habitatge. En cas d'escollir lloguer, preguntar a l'usuari els ingressos mensuals de la seua llar, i només mostrar aquells que estiguen per sota del 35%.

3. **Planta:** Preguntar als usuaris a quina planta volen viure. Al final de l'execució, només s'oferiràn habitatges que estiguen en aquella planta o superior.

4. **Terrassa:** Preguntar a l'usuari si vol un habitatge amb terrassa o no. En cas de voler terrassa, només s'oferiran habitatges amb terrassa. En cas contrari, oferir habitatges amb terrassa o sense.

5. **Ascensor:** Preguntar a l'usuari si vol un habitatge amb ascensor o no. En cas de voler ascensor, només s'oferiran habitatges amb ascensor. En cas contrari, oferir habitatges amb ascensor o sense.

6. **Us comercial:** Preguntar a l'usuari si vol fer servir l'habitatge com a negoci. En cas afirmatiu, només oferir habitatges amb la propietat *"commercial use"*

---





In [24]:
# Gestió de les respostes (volem respostes vàlides)
def read_answer(question, validator):
    while True:
        answer = input(question['prompt']).strip()
        lowered = answer.lower()

        if lowered in {"q", "quit"}:
            return "quit"
        if lowered == "any":
            return "any"

        value = validator(answer)
        if value is not None:
            return value


def numeric_value_from_text(value):
    try:
        return float(str(value).lower().replace('k', '000'))
    except ValueError:
        return None


def numeric_value(value):
    return numeric_value_from_text(value)

NUMERIC_RANGES = {
    'bedrooms': (1, 5),
    'bathrooms': (1, 3),
    'square_meters': (30, 200),
    'income': (0, float('inf')),
    'floor': (0, float('inf'))
}

def is_numeric_value_in_range(answer_key, value, house_type=None):
    if answer_key == 'price':
        minimum, maximum = (500, 5000) if house_type == 'rent' else (100000, 1000000)
    elif answer_key in NUMERIC_RANGES:
        minimum, maximum = NUMERIC_RANGES[answer_key]
    else:
        return True

    numeric_answer = numeric_value_from_text(value)
    return numeric_answer is not None and minimum <= numeric_answer <= maximum


def process_numerical_question(question, house_type=None):
    print_question(question['question'])

    def validator(answer):
        value = get_numerical_value(preprocess_answer(answer))
        if value == '':
            return None
        if is_numeric_value_in_range(question['answer_key'], value, house_type):
            return value
        return None

    return read_answer(question, validator)


def process_multichoice_question(question, options):
    print_question(question['question'], options)
    normalized_options = {option.lower(): option for option in options}

    def validator(answer):
        return normalized_options.get(answer.lower())

    return read_answer(question, validator)


def process_yes_no_question(question):
    return process_multichoice_question(question, ["Yes", "No"])

In [25]:
# Gestió del chatbot

def ask_or_quit(question_data, options=None, house_type=None):
    """Fa una pregunta i surt del programa si l'usuari respon 'quit'."""
    if options is not None:
        answer = process_multichoice_question(question_data, options)
    else:
        answer = process_numerical_question(question_data, house_type)

    if answer == "quit":
            print(data['end_message'])
            sys.exit()

    return answer


def ask_numeric(question, house_type):
    return ask_or_quit(question, house_type=house_type)


def ask_user_preferences():
    user_preferences, available_options = {}, {}
    initialize_available_options(data, available_options)

    user_preferences['type'] = ask_or_quit(
        {
            "question": "Do you want to buy or rent a house?",
            "prompt": "Enter sale or rent: "
        },
        options=["sale", "rent"]
    )

    for question in data['questions']:
        answer_key = question['answer_key']
        options = sorted(available_options.get(answer_key, set()))

        if question['type'] == 'numerical':
            answer = ask_numeric(question, user_preferences['type'])
        else:
            answer = ask_or_quit(question, options=options)

        user_preferences[answer_key] = answer

    if user_preferences['type'] == "rent":
        user_preferences['income'] = ask_numeric(
            {
                "question": "What is your monthly household income?",
                "prompt": "Enter your monthly income in euros: ",
                "answer_key": "income"
            },
            user_preferences['type']
        )

    user_preferences['floor'] = ask_numeric(
        {
            "question": "What is the minimum floor where you want to live?",
            "prompt": "Enter the minimum floor (0 or higher): ",
            "answer_key": "floor"
        },
        user_preferences['type']
    )

    optional_questions = {
        'terrace': "Do you need a terrace?",
        'elevator': "Do you need an elevator?",
        'commercial_use': "Will you use the house for commercial purposes?"
    }
    for answer_key, question_text in optional_questions.items():
        user_preferences[answer_key] = ask_or_quit(
            {"question": question_text, "prompt": "Enter Yes or No: "},
            options=["Yes", "No"]
        )

    return user_preferences

In [31]:
# Funcions relacionades amb la casa perfecte per tu
def find_suitable_houses(data, user_preferences):
    suitable_houses = []
    requested_type = user_preferences.get('type')
    minimum_floor = float(user_preferences.get('floor', 0))
    income = numeric_value(user_preferences['income']) if 'income' in user_preferences else None

    for house in data['houses']:
        if house.get('type') != requested_type:
            continue
        if numeric_value(house.get('floor', 0)) < minimum_floor:
            continue
        if income is not None and numeric_value(house['price']) > income * 0.35:
            continue

        matches_preferences = all(
            value == 'any'
            or key in {'type', 'income', 'floor'}
            or value == 'No'
            or house.get(key) == value
            for key, value in user_preferences.items()
        )

        if matches_preferences:
            suitable_houses.append(house)

    return suitable_houses


def print_suitable_houses(suitable_houses):
  if suitable_houses:
    print("\nBased on your preferences, the most suitable houses are:")
    for house in suitable_houses:
      print("\nHouse characteristics:")
      for key, value in house.items():
        label = key.replace('_', ' ').capitalize()
        print(f"{label}: {value}")
      print()
  else:
    print("\nSorry, no suitable houses match your preferences. \n")

In [32]:
def chatbot_v5():
  print(data['start_message'])
  user_preferences = ask_user_preferences()
  suitable_houses = find_suitable_houses(data, user_preferences)
  print_suitable_houses(suitable_houses)
  print(data['end_message'])

In [33]:
chatbot_v5()

Welcome to the House Buying Assistant!
Do you want to buy or rent a house?
Options: sale, rent
How many bedrooms do you need?
How many bathrooms do you need?
What is your budget for the house?
How many square meters do you need?
Which city or neighborhood would you prefer?
Options: Barcelona, Esplugues de Llobregat, L'Hospitalet de Llobregat, Montcada i Reixac, Santa Coloma de Gramenet
What is your monthly household income?
What is the minimum floor where you want to live?
Do you need a terrace?
Options: Yes, No
Do you need an elevator?
Options: Yes, No
Will you use the house for commercial purposes?
Options: Yes, No

Based on your preferences, the most suitable houses are:

House characteristics:
Id: 1
Type: rent
Bedrooms: 3
Bathrooms: 2
Price: 1200
Square meters: 100
Floor: 3
Elevator: Yes
Commercial use: No
Terrace: No
Location: Santa Coloma de Gramenet

Thank you for using the House Buying Assistant. Goodbye!


 <h1><a name="ex-seven"><center> ✏ Exercici 7 ✏ </a></h1>

Ara ja teniu un sistema funcional. Aplicant el mètode vist a classe (Exemples, Camins, Prototipat i Proves) milloreu aquest sistema per donar una millor experiència als usuaris.

---